3. Feature Engineerig

In [3]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

import pandas as pd
import numpy as np
IN, OUT = ROOT/'data/processed/tasks_cleaned.csv', ROOT/'data/processed'
t = pd.read_csv(IN, parse_dates=['created','target','status_changed'])

t['days_to_target'] = (t['target'] - t['created']).dt.days
t['month'] = t['created'].dt.month
t['weekday'] = t['created'].dt.weekday
t['season'] = ((t['month'] % 12 + 3) // 3).map({1:'winter',2:'spring',3:'summer',4:'autumn'})
t['has_parent_task'] = t['association'].eq('ForwardedFrom').astype('int8')
t['location_depth'] = t['location'].fillna('').str.count('>') + 1
t['description_length'] = t['description'].fillna('').str.len()
t['tasks_per_project'] = t.groupby('project')['ref'].transform('size')
t['issues_per_project'] = t.groupby('project')['overdue'].transform('sum')
t['task_category'] = t['task_group'].fillna('Unknown')
# days_open uses status_changed and is for retrospective reporting only. I will exclude it from predictive features. The reason for this decision is that this feature is known after the task is done not before that.
t['days_open'] = (t['status_changed'] - t['created']).dt.days
t.to_csv(OUT/'features.csv', index=False)
t[['days_to_target','month','weekday','season','has_parent_task','location_depth','description_length','tasks_per_project','issues_per_project','days_open']].describe(include='all')



,days_to_target,month,weekday,season,has_parent_task,location_depth,description_length,tasks_per_project,issues_per_project,days_open
count,2568.000000,12424.000000,12424.000000,12424,12424.000000,12424.000000,12424.000000,12424.000000,12424.000000,12424.000000
unique,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,summer,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,6086,NaN,NaN,NaN,NaN,NaN,NaN
mean,6.053349,6.490019,2.303203,NaN,0.015534,2.799340,54.644237,2626.793303,171.377898,11.055699
std,5.646573,2.675488,1.389663,NaN,0.123670,1.151586,45.998502,1350.153907,82.356131,28.246425
min,-86.000000,1.000000,0.000000,NaN,0.000000,1.000000,1.000000,381.000000,0.000000,0.000000
25%,4.000000,5.000000,1.000000,NaN,0.000000,2.000000,26.000000,1267.000000,192.000000,0.000000
50%,6.000000,7.000000,2.000000,NaN,0.000000,3.000000,41.000000,3684.000000,205.000000,1.000000
75%,6.000000,8.000000,3.000000,NaN,0.000000,3.000000,68.000000,3751.000000,226.000000,8.000000
